In [916]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 

In [917]:
df = pd.read_csv("../../preprocessed_dataset.csv")
print(df.iloc[1])

Age                                             49
Attrition                                       No
BusinessTravel                   Travel_Frequently
DailyRate                                      279
Department                  Research & Development
DistanceFromHome                                 8
Education                            Below College
EducationField                       Life Sciences
EnvironmentSatisfaction                       high
Gender                                        Male
HourlyRate                                      61
JobInvolvement                              medium
JobLevel                                         2
JobRole                         Research Scientist
JobSatisfaction                                  2
MaritalStatus                              Married
MonthlyIncome                                 5130
MonthlyRate                                  24907
NumCompaniesWorked                               1
OverTime                       

In [918]:
from sklearn.model_selection import train_test_split

x = df.drop(["Attrition"], axis = 1)
y = df["Attrition"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3,stratify=y, random_state=42)

In [919]:
y_train[y_train == "Yes"].count()/ len(y_train) # imbalanced dataset 

np.float64(0.16132167152575316)

In [920]:
num_col = ["Age", "DailyRate","DistanceFromHome", "HourlyRate", "MonthlyIncome", "MonthlyRate", "NumCompaniesWorked", "PercentSalaryHike", "YearsWithCurrManager", "YearsInCurrentRole","TrainingTimesLastYear","YearsSinceLastPromotion", "YearsAtCompany", "TotalWorkingYears"]
len(num_col)

ohe = ["MaritalStatus", "JobRole", "Department", "Gender", "EducationField", "OverTime", "StockOptionLevel"]
len(ohe)

ordinal = list(set(x_train.columns) - set(num_col) - set(ohe))

In [921]:
dict1 = {}
for i in ordinal:
    df[i].unique().tolist().sort()
    k=df[i].unique().tolist()
    print(f"{k} : {i}")

dict1

['medium', 'high', 'very high', 'low'] : EnvironmentSatisfaction
['low', 'very high', 'medium', 'high'] : RelationshipSatisfaction
[4, 2, 3, 1] : JobSatisfaction
[2, 1, 3, 4, 5] : JobLevel
['College', 'Below College', 'Master', 'Bachelor', 'Doctor'] : Education
['Travel_Rarely', 'Travel_Frequently', 'Non-Travel'] : BusinessTravel
['bad', 'better', 'good', 'best'] : WorkLifeBalance
['high', 'medium', 'very high', 'low'] : JobInvolvement
['excellent', 'outstanding'] : PerformanceRating


{}

In [922]:
ordinal_categories = [
    ["low", "medium", "high", "very high"],
    ["low", "medium", "high", "very high"],
    ["low", "medium", "high", "very high"],
    ["Below College", "College", "Bachelor", "Master", "Doctor"],
    ["bad", "better", "good", "best"],
    ["excellent", "outstanding"]
]

In [923]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

transformer = ColumnTransformer([
    ("trf1", StandardScaler(), ["Age", "DailyRate","DistanceFromHome", "HourlyRate", "MonthlyIncome", "MonthlyRate", "NumCompaniesWorked", "PercentSalaryHike", "YearsWithCurrManager", "YearsInCurrentRole","TrainingTimesLastYear","YearsSinceLastPromotion",  "YearsAtCompany", "TotalWorkingYears"]),
    ("trf2", OrdinalEncoder(categories = ordinal_categories), ["EnvironmentSatisfaction", "RelationshipSatisfaction", "JobInvolvement", "Education", "WorkLifeBalance", "PerformanceRating"]),
    ("trf3", OneHotEncoder(drop = "first", sparse_output=False), ["MaritalStatus", "JobRole", "Department", "Gender", "EducationField", "OverTime", "StockOptionLevel", "BusinessTravel"])
], remainder="passthrough")

In [924]:
from sklearn.model_selection import train_test_split

x_test, x_val, y_test, y_val = train_test_split(
    x_test, y_test,
    test_size=0.5,
    random_state=42,
    stratify=y_test
)

In [925]:
x_train = transformer.fit_transform(x_train)
x_val = transformer.transform(x_val)
x_test = transformer.transform(x_test)

In [926]:
x_train.shape

(1029, 46)

In [927]:
from imblearn.over_sampling import SMOTE

smote = SMOTE()
x_train, y_train = smote.fit_resample(x_train, y_train)


In [928]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

In [929]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

def metrics(y_test, pred):
    print("Accuracy_Score = ", accuracy_score(y_test, pred))
    print("Precision Score = ", precision_score(y_test, pred))
    print("Recall Score = ", recall_score(y_test, pred))
    
    print("f1 score = ", f1_score(y_test, pred))
    print("Confusion Matrix\n", confusion_matrix(y_test, pred))

In [930]:
def model_run(model):
    model.fit(x_train, y_train)
    pred = model.predict(x_train)
    print("Training metrics")
    metrics(y_train, pred)
    y_prob = model.predict_proba(x_train)[:, 1]
    print("ROC-AUC area = ",roc_auc_score(y_train, y_prob))

    print("\nValidation metrics")
    pred_val = model.predict(x_val)
    metrics(y_val, pred_val)
    y_prob = model.predict_proba(x_val)[:, 1]
    print("ROC-AUC area = ", roc_auc_score(y_val, y_prob))

In [931]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=350)
model_run(model_lr)

Training metrics
Accuracy_Score =  0.8227114716106605
Precision Score =  0.8212226066897347
Recall Score =  0.8250289687137891
f1 score =  0.823121387283237
Confusion Matrix
 [[708 155]
 [151 712]]
ROC-AUC area =  0.9012458896651175

Validation metrics
Accuracy_Score =  0.7239819004524887
Precision Score =  0.3287671232876712
Recall Score =  0.6666666666666666
f1 score =  0.44036697247706424
Confusion Matrix
 [[136  49]
 [ 12  24]]
ROC-AUC area =  0.8150150150150149


In [932]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(max_depth = 5)
model_run(model_dt)

Training metrics
Accuracy_Score =  0.8835457705677868
Precision Score =  0.9168765743073047
Recall Score =  0.8435689455388181
f1 score =  0.8786964393482197
Confusion Matrix
 [[797  66]
 [135 728]]
ROC-AUC area =  0.9411784056532965

Validation metrics
Accuracy_Score =  0.8280542986425339
Precision Score =  0.47368421052631576
Recall Score =  0.5
f1 score =  0.4864864864864865
Confusion Matrix
 [[165  20]
 [ 18  18]]
ROC-AUC area =  0.6868618618618618


In [933]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier(n_neighbors = 5, n_jobs=-1)
model_run(model_knn)

Training metrics
Accuracy_Score =  0.8568945538818077
Precision Score =  0.78
Recall Score =  0.9942062572421785
f1 score =  0.8741721854304636
Confusion Matrix
 [[621 242]
 [  5 858]]
ROC-AUC area =  0.9940719874216033

Validation metrics
Accuracy_Score =  0.6153846153846154
Precision Score =  0.25252525252525254
Recall Score =  0.6944444444444444
f1 score =  0.37037037037037035
Confusion Matrix
 [[111  74]
 [ 11  25]]
ROC-AUC area =  0.6715465465465464


In [934]:
from sklearn.svm import SVC

model_svm = SVC(kernel="rbf", gamma="auto", C = 2, probability=True)
model_run(model_svm)

c:\Users\Nanda kishore\Desktop\coding\projects\IBM attrition project\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Training metrics
Accuracy_Score =  0.9611819235225956
Precision Score =  0.9451901565995525
Recall Score =  0.9791425260718424
f1 score =  0.9618668184405236
Confusion Matrix
 [[814  49]
 [ 18 845]]
ROC-AUC area =  0.9901123704128394

Validation metrics
Accuracy_Score =  0.8280542986425339
Precision Score =  0.48
Recall Score =  0.6666666666666666
f1 score =  0.5581395348837209
Confusion Matrix
 [[159  26]
 [ 12  24]]
ROC-AUC area =  0.7954954954954956


In [935]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(max_depth=10)
model_run(model_rf)

Training metrics
Accuracy_Score =  0.9994206257242179
Precision Score =  1.0
Recall Score =  0.9988412514484357
f1 score =  0.9994202898550725
Confusion Matrix
 [[863   0]
 [  1 862]]
ROC-AUC area =  1.0

Validation metrics
Accuracy_Score =  0.8371040723981901
Precision Score =  0.5
Recall Score =  0.25
f1 score =  0.3333333333333333
Confusion Matrix
 [[176   9]
 [ 27   9]]
ROC-AUC area =  0.8364864864864865


In [936]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(max_depth = 10)
model_run(model_xgb)

Training metrics
Accuracy_Score =  1.0
Precision Score =  1.0
Recall Score =  1.0
f1 score =  1.0
Confusion Matrix
 [[863   0]
 [  0 863]]
ROC-AUC area =  1.0

Validation metrics
Accuracy_Score =  0.8868778280542986
Precision Score =  0.7619047619047619
Recall Score =  0.4444444444444444
f1 score =  0.5614035087719298
Confusion Matrix
 [[180   5]
 [ 20  16]]
ROC-AUC area =  0.792042042042042


In [937]:
from sklearn.naive_bayes import GaussianNB

model_nb = GaussianNB()
model_run(model_nb)

Training metrics
Accuracy_Score =  0.6807647740440325
Precision Score =  0.634020618556701
Recall Score =  0.8551564310544612
f1 score =  0.7281697089294524
Confusion Matrix
 [[437 426]
 [125 738]]
ROC-AUC area =  0.8088722812039706

Validation metrics
Accuracy_Score =  0.5520361990950227
Precision Score =  0.23076923076923078
Recall Score =  0.75
f1 score =  0.35294117647058826
Confusion Matrix
 [[95 90]
 [ 9 27]]
ROC-AUC area =  0.7124624624624625


In [938]:
from sklearn.ensemble import AdaBoostClassifier

model_ada = AdaBoostClassifier(learning_rate=0.01)
model_run(model_ada)

Training metrics
Accuracy_Score =  0.7207415990730012
Precision Score =  0.7343173431734318
Recall Score =  0.6917728852838934
f1 score =  0.7124105011933174
Confusion Matrix
 [[647 216]
 [266 597]]
ROC-AUC area =  0.8397872360423164

Validation metrics
Accuracy_Score =  0.7601809954751131
Precision Score =  0.3508771929824561
Recall Score =  0.5555555555555556
f1 score =  0.43010752688172044
Confusion Matrix
 [[148  37]
 [ 16  20]]
ROC-AUC area =  0.7373123123123122


In [939]:
from sklearn.neural_network import MLPClassifier

model_mlp = MLPClassifier(hidden_layer_sizes=(1024, 256), activation="relu",learning_rate="adaptive", early_stopping=True, solver = "adam", max_iter=300)
model_run(model_mlp)

Training metrics
Accuracy_Score =  0.9982618771726536
Precision Score =  0.9965357967667436
Recall Score =  1.0
f1 score =  0.9982648930017352
Confusion Matrix
 [[860   3]
 [  0 863]]
ROC-AUC area =  0.9999207808058606

Validation metrics
Accuracy_Score =  0.8642533936651584
Precision Score =  0.59375
Recall Score =  0.5277777777777778
f1 score =  0.5588235294117647
Confusion Matrix
 [[172  13]
 [ 17  19]]
ROC-AUC area =  0.7755255255255256


In [940]:
def testing_metrics(model):
    pred_test = model.predict(x_test)
    metrics(y_test, pred_test)
    y_prob = model.predict_proba(x_test)[:, 1]
    print("ROC-AUC area = ", roc_auc_score(y_test, y_prob))

    

In [941]:
testing_metrics(model_mlp)

Accuracy_Score =  0.8681818181818182
Precision Score =  0.6363636363636364
Recall Score =  0.4
f1 score =  0.49122807017543857
Confusion Matrix
 [[177   8]
 [ 21  14]]
ROC-AUC area =  0.7484169884169883


In [942]:
testing_metrics(model_xgb)

Accuracy_Score =  0.8318181818181818
Precision Score =  0.4444444444444444
Recall Score =  0.22857142857142856
f1 score =  0.3018867924528302
Confusion Matrix
 [[175  10]
 [ 27   8]]
ROC-AUC area =  0.7150579150579149


In [943]:
testing_metrics(model_rf)

Accuracy_Score =  0.8454545454545455
Precision Score =  0.5454545454545454
Recall Score =  0.17142857142857143
f1 score =  0.2608695652173913
Confusion Matrix
 [[180   5]
 [ 29   6]]
ROC-AUC area =  0.7079536679536679


In [944]:
testing_metrics(model_svm)

Accuracy_Score =  0.8409090909090909
Precision Score =  0.5
Recall Score =  0.4857142857142857
f1 score =  0.4927536231884058
Confusion Matrix
 [[168  17]
 [ 18  17]]
ROC-AUC area =  0.7609266409266409


In [945]:
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier(
    estimators=[
        ("mlp", model_mlp),
        ("xgb", model_xgb),
        ("rf", model_rf),
        ("svc", model_svm)
    ],
    voting="soft"
)

voting_model.fit(x_train, y_train)

c:\Users\Nanda kishore\Desktop\coding\projects\IBM attrition project\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('mlp', ...), ('xgb', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](2,)","[0,1]"
"estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators as defined in ``estimators``that are not 'drop'. Note that sub-estimators are always fitted oninteger-encoded labels (see ``le_`` attribute). When ``y`` containsnon-integer class labels (e.g. strings), use ``le_.inverse_transform``to map predictions back to the original label space.",list,"[MLPClassifier... max_iter=300), XGBClassifier...ree=None, ...), RandomForestC...(max_depth=10), SVC(C=2, gamm...bability=True)]"
le_ le_: :class:`~sklearn.preprocessing.LabelEncoder`Transformer used to encode the labels during fit and decode duringprediction. Sub-estimators in ``estimators_`` are fitted on theinteger-encoded labels produced by this encoder.,LabelEncoder,LabelEncoder()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying classifier exposes such an attribute when fit... versionadded:: 0.24,int,46


In [946]:
testing_metrics(voting_model)

Accuracy_Score =  0.8681818181818182
Precision Score =  0.6666666666666666
Recall Score =  0.34285714285714286
f1 score =  0.4528301886792453
Confusion Matrix
 [[179   6]
 [ 23  12]]
ROC-AUC area =  0.7521235521235521


In [951]:
import pickle

def write_model(name, model):
    with open(name, "wb") as f:
        pickle.dump(model, f)

    return True

In [952]:
write_model(name = "../../transformers.pkl", model = transformer)

True

In [954]:
files = ["model_mlp.pkl", "model_svm.pkl", "model_rf.pkl", "model_xgb.pkl", "voting_model.pkl"]
models = [model_mlp, model_svm, model_rf, model_xgb, voting_model]

for file, model in zip(files, models):
    file = "../../" + file
    status = write_model(name = file, model = model)
    if status:
        print(model, " converted to .pkl file successfully")

MLPClassifier(early_stopping=True, hidden_layer_sizes=(1024, 256),
              learning_rate='adaptive', max_iter=300)  converted to .pkl file successfully
SVC(C=2, gamma='auto', probability=True)  converted to .pkl file successfully
RandomForestClassifier(max_depth=10)  converted to .pkl file successfully
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estim

In [955]:
write_model(name = "../../label_encoder.pkl", model = le)

True